# CPSC 425 Assignment 1: Image Filtering and Hybrid Images

**Name:**

**Student number:**

All the images I used (`dog.jpg`, the hybrid image pairs, `box_gauss.png` and `box_speckle.png`) have to be in the same folder as this notebook (on Colab I just uploaded them in the Files tab on the left).

Part 1 (the written questions) is in a separate PDF.

In [ ]:
# imports, done exactly the way the assignment asks
from PIL import Image
import numpy as np
import math
from scipy import signal
import cv2

# for showing images in jupyter / colab
from IPython.display import display

# only needed for the timing question (part 2 q6)
import time

In [ ]:
# ---- a couple of helper functions I use all over the notebook ----

def to_uint8(arr):
    '''Turns a float image array back into uint8 so PIL can show / save it.
    I clip to 0-255 first because otherwise values like 260 or -5 wrap around
    when you cast to uint8 (260 -> 4), which gives weird coloured pixels.'''
    return np.clip(np.round(arr), 0, 255).astype(np.uint8)


def show_row(images, gap=10):
    '''Shows a list of PIL images next to each other (left to right) as one picture.
    Mostly so the pdf doesn't get super long from calling display() on every image.'''
    total_w = sum(im.width for im in images) + gap * (len(images) - 1)
    max_h = max(im.height for im in images)
    # white background, and RGB so greyscale + colour images can go in the same row
    canvas = Image.new('RGB', (total_w, max_h), (255, 255, 255))
    x = 0
    for im in images:
        canvas.paste(im.convert('RGB'), (x, 0))
        x += im.width + gap
    display(canvas)

# Part 2: Gaussian Filtering

## Q1. Box filter

In [ ]:
def boxfilter(n):
    '''Returns an n x n box filter (numpy array).
    Every entry is 1/(n*n) so everything adds up to 1, i.e. it just averages the neighbourhood.
    n has to be odd, if it isn't an AssertionError is raised.'''
    # course convention: filters have an odd size so there is an actual center pixel
    assert n % 2 == 1, "Dimension must be odd"
    return np.full((n, n), 1.0 / (n * n))

In [ ]:
print("boxfilter(3):")
print(boxfilter(3))

print("\nboxfilter(4):")
# 4 is even so this one should fail.
# I catch the error here so the rest of the notebook keeps running
try:
    print(boxfilter(4))
except AssertionError as e:
    print("AssertionError:", e)

print("\nboxfilter(5):")
print(boxfilter(5))

## Q2. 1D Gaussian filter

In [ ]:
def gauss1d(sigma):
    '''Returns a 1D gaussian filter (1D numpy array) for the given sigma.
    The length is 6*sigma rounded up to the next odd integer and the values
    are normalized so they sum to 1.'''
    # length of the filter: round 6*sigma up, and if that ends up even go to the next odd number
    length = math.ceil(6 * sigma)
    if length % 2 == 0:
        length += 1

    # x = distance of each entry from the center, e.g. length 11 -> [-5 -4 ... 4 5]
    half = length // 2
    x = np.arange(-half, half + 1, dtype=np.float64)

    # put all the x values through exp(-x^2 / (2 sigma^2)) at once (no for loop needed)
    g = np.exp(-x ** 2 / (2 * sigma ** 2))

    # we left out the constant in front of the gaussian, so normalize here instead
    return g / np.sum(g)

In [ ]:
# filter values for the sigmas the assignment asks for
for s in [0.3, 0.5, 1, 2]:
    g = gauss1d(s)
    print(f"sigma = {s}  (length {len(g)}):")
    print(g)
    print()

## Q3. 2D Gaussian filter

In [ ]:
def gauss2d(sigma):
    '''Returns a 2D gaussian filter (square 2D numpy array) for the given sigma.
    It's made by convolving the 1D gaussian with its transpose using scipy's convolve2d.'''
    # convolve2d only takes 2D arrays, so turn the 1D filter into a 1 x n row first
    g = gauss1d(sigma)[np.newaxis]
    # (1 x n) row convolved with (n x 1) column = the full n x n filter
    # default mode is 'full', which is what I want here so the output is n x n
    return signal.convolve2d(g, g.T)

In [ ]:
for s in [0.5, 1]:
    print(f"gauss2d({s}):")
    print(gauss2d(s))
    print()

## Q4. Convolution by hand

### (a) `convolve2d_manual`

In [ ]:
def convolve2d_manual(array, filter):
    '''Convolves a 2D image (array) with a 2D filter, using zero padding so the
    output is the same size as the input. Both inputs should be np.float32.

    Uses two for loops (rows and columns). For every pixel I cut out the
    neighbourhood around it, multiply it elementwise with the flipped filter and sum.'''
    # convolution = correlation but with the filter flipped both up/down and left/right.
    # (for a gaussian this doesn't change anything since it's symmetric, but I want this
    #  to be an actual convolution so it works for any filter)
    kernel = np.flipud(np.fliplr(filter)).astype(np.float32)
    k_h, k_w = kernel.shape
    pad_h, pad_w = k_h // 2, k_w // 2

    # zero padding around the image so the filter can also sit on the border pixels
    padded = np.pad(array.astype(np.float32), ((pad_h, pad_h), (pad_w, pad_w)),
                    mode='constant', constant_values=0)

    rows, cols = array.shape
    out = np.zeros((rows, cols), dtype=np.float32)

    for i in range(rows):
        for j in range(cols):
            # because of the padding, padded[i:i+k_h, j:j+k_w] is the neighbourhood
            # centered at pixel (i, j) of the original image
            patch = padded[i:i + k_h, j:j + k_w]
            out[i, j] = np.sum(patch * kernel)

    return out

In [ ]:
# quick sanity check before using it on the real image:
# compare against scipy on a small random array with a filter that is NOT symmetric
# (if I forgot to flip the filter this would fail)
test_arr = np.random.rand(7, 9).astype(np.float32)
test_filt = np.random.rand(3, 5).astype(np.float32)
mine = convolve2d_manual(test_arr, test_filt)
theirs = signal.convolve2d(test_arr, test_filt, 'same')
print("same as scipy convolve2d?", np.allclose(mine, theirs, atol=1e-5))

### (b) `gaussconvolve2d_manual`

In [ ]:
def gaussconvolve2d_manual(array, sigma):
    '''Gaussian blur of a 2D array: make the filter with gauss2d and then
    apply it with my own convolve2d_manual. Returns a 2D array (same size as input).'''
    filt = gauss2d(sigma).astype(np.float32)
    return convolve2d_manual(array.astype(np.float32), filt)

### (c) Blurring the dog with sigma = 3

In [ ]:
# load the dog, convert to greyscale ('L'), then to a float numpy array
# (doing the filtering in float avoids rounding / overflow problems with uint8)
dog = Image.open('dog.jpg')
dog_grey = dog.convert('L')
dog_arr = np.asarray(dog_grey, dtype=np.float32)
print("dog image size (rows, cols):", dog_arr.shape)

dog_blur_manual = gaussconvolve2d_manual(dog_arr, 3)

# done filtering -> back to uint8 so we can display / save it
dog_blur_manual_img = Image.fromarray(to_uint8(dog_blur_manual))
dog_blur_manual_img.save('dog_blur_manual.png')

### (d) Original vs. filtered

In [ ]:
print("original image:")
display(dog)

print("left: greyscale version,  right: blurred with sigma = 3 (gaussconvolve2d_manual)")
show_row([dog_grey, dog_blur_manual_img])

(The dark frame around the blurred image comes from the zero padding. Near the border part of the filter sits on the padded 0s, so those pixels get pulled towards black.)

## Q5. Convolution with scipy

### (a) `gaussconvolve2d_scipy`

In [ ]:
def gaussconvolve2d_scipy(array, sigma):
    '''Same thing as gaussconvolve2d_manual but using scipy's convolve2d.
    'same' makes the output the same size as the input (it zero pads too).'''
    filt = gauss2d(sigma)
    return signal.convolve2d(array, filt, 'same')

**Why does scipy have both `signal.convolve2d` and `signal.correlate2d`?**

Convolution and correlation are almost the same operation. The only difference is that convolution flips the filter (up/down and left/right) before sliding it over the image, and correlation uses the filter as is. The gaussian from `gauss2d` is symmetric, so flipping it gives the exact same filter, and that's why both functions gave the same result here.

They give different results as soon as the filter is not symmetric, for example a derivative filter like `[-1, 0, 1]` (the sign of the output flips) or a filter that shifts the image (convolution moves the image one way and correlation moves it the other way, see the small example below).

So they are separate because they are useful for different things. Convolution is the "real" linear filtering operation and has nice math properties (it's commutative and associative, so e.g. filters can be combined first, which is exactly how we built `gauss2d`). Correlation doesn't have those properties, but it's what you want for things like template matching, where you want to compare the image with the filter the way it is, without flipping it.

In [ ]:
# small example with a filter that is not symmetric (a 1 on the right of the center)
test_img = np.arange(1, 26, dtype=np.float32).reshape(5, 5)
shift_filt = np.array([[0, 0, 0],
                       [0, 0, 1],
                       [0, 0, 0]], dtype=np.float32)

print("test image:")
print(test_img)
print("\nconvolve2d  -> image moves one pixel to the RIGHT:")
print(signal.convolve2d(test_img, shift_filt, 'same'))
print("\ncorrelate2d -> image moves one pixel to the LEFT:")
print(signal.correlate2d(test_img, shift_filt, 'same'))

### (b) Blurring the dog again with scipy (sigma = 3)

In [ ]:
# same greyscale float array as in Q4 (c)
dog_blur_scipy = gaussconvolve2d_scipy(dog_arr, 3)

dog_blur_scipy_img = Image.fromarray(to_uint8(dog_blur_scipy))
dog_blur_scipy_img.save('dog_blur_scipy.png')

# should be basically identical to my manual version (just float rounding differences)
print("max difference between manual and scipy result:",
      np.max(np.abs(dog_blur_scipy - dog_blur_manual)))

### (c) Original vs. filtered

In [ ]:
print("left: original (greyscale),  right: blurred with sigma = 3 (gaussconvolve2d_scipy)")
show_row([dog_grey, dog_blur_scipy_img])

## Q6. Timing: manual vs. scipy (sigma = 10)

In [ ]:
sigma = 10.0
print("filter size for sigma = 10:", gauss2d(sigma).shape)

# my version with the two for loops
t1 = time.time()
blur_manual_10 = gaussconvolve2d_manual(dog_arr, sigma)
duration_manual = time.time() - t1

# scipy version
t1 = time.time()
blur_scipy_10 = gaussconvolve2d_scipy(dog_arr, sigma)
duration_scipy = time.time() - t1

print(f"gaussconvolve2d_manual: {duration_manual:.3f} s")
print(f"gaussconvolve2d_scipy:  {duration_scipy:.3f} s")
print(f"manual / scipy: {duration_manual / duration_scipy:.1f}x")

# also timing a few other sigmas to see how the gap changes with the filter size
print("\nsigma | filter size | manual (s) | scipy (s) | manual / scipy")
for s in [1, 3, 5, 10]:
    t1 = time.time()
    gaussconvolve2d_manual(dog_arr, s)
    t_m = time.time() - t1
    t1 = time.time()
    gaussconvolve2d_scipy(dog_arr, s)
    t_s = time.time() - t1
    n = gauss2d(s).shape[0]
    print(f"{s:5} | {n:4} x {n:<4} | {t_m:10.3f} | {t_s:9.3f} | {t_m / t_s:.1f}x")

**Comments:**

With sigma = 10 the filter is 61 x 61, so both functions have to do 61 * 61 = 3721 multiply-adds for every pixel. I expected scipy to be way faster, but at sigma = 10 the two times were surprisingly close (the exact numbers moved around a bit between runs). That's why I also timed some smaller sigmas (table above). My manual version takes roughly the same time no matter what sigma is (until the filter gets really big), while scipy keeps getting slower as sigma goes up. For small sigmas scipy is way faster.

Why this happens: in `convolve2d_manual` the two for loops over the pixels run in plain Python. Every iteration has a fixed cost (the Python loop itself, slicing out the patch, making a temporary array for `patch * kernel`, calling `np.sum`) that doesn't really depend on the filter size, and with one iteration per pixel that adds up to a pretty constant amount of time. The actual multiplying and adding inside a patch is done by numpy in C, so that part is fast. Scipy's `convolve2d` does everything, loops included, in compiled C code, so it has no per-pixel Python overhead at all. But it's still a direct convolution (it doesn't use FFT or anything), so its time grows with the size of the filter, i.e. with sigma squared.

So with a small filter scipy only has a little bit of work to do and the Python overhead in my version dominates, and scipy wins by a lot. With sigma = 10 most of the time goes into the 3721 multiplications per pixel, which both versions do in C, so they end up much closer. Either way, both do way more work than needed for a big gaussian (see Q7).

## Q7. Using separability

A 2D gaussian is separable: it's just a 1D gaussian (row) convolved with the same 1D gaussian as a column, which is exactly how `gauss2d` builds it. Since convolution is associative, blurring the image with the 2D filter gives the same result as first convolving the image with the 1D row filter (blurs horizontally) and then convolving that result with the 1D column filter (blurs vertically).

This is faster because of the number of operations per pixel. With an n x n filter, the 2D version does n^2 multiplications per pixel, while the separable version does n for the row pass and n for the column pass, so 2n in total. For sigma = 10 (n = 61) that's 3721 vs. 122 multiplications per pixel, so about 30 times less work, and the savings get bigger the larger sigma is (O(n^2) vs. O(n) per pixel).

# Part 3: Hybrid Images

For colour images I just run `gaussconvolve2d_scipy` on the R, G and B channels separately (same sigma for all of them) and stack them back together. I used the scipy version and not my manual one because it's a lot faster and gives the same result (see Q5).

The first image in each pair gives the low frequencies and the second one gives the high frequencies.

In [ ]:
# the pairs I used: (image for low frequencies, image for high frequencies)
# file names are the ones from the hybrid images folder on the course page
pairs = [
    ('0a_dog.bmp', '0b_cat.bmp'),
    ('1a_bicycle.bmp', '1b_motorcycle.bmp'),
    ('2a_einstein.bmp', '2b_marilyn.bmp'),
]


def load_rgb(path):
    '''Loads an image as an RGB float32 array (rows x cols x 3).'''
    return np.asarray(Image.open(path).convert('RGB'), dtype=np.float32)


def low_pass(img, sigma):
    '''Gaussian blur on a colour image: blur each of the 3 channels on its own
    with the same sigma, then put them back together. Returns a float array.'''
    out = np.zeros(img.shape, dtype=np.float32)
    for c in range(3):  # 0 = R, 1 = G, 2 = B
        out[:, :, c] = gaussconvolve2d_scipy(img[:, :, c], sigma)
    return out


def high_pass(img, sigma):
    '''High frequencies = original - blurred (low frequencies), per channel.
    The result is roughly zero mean and has negative values, so it's still a float array.'''
    return img - low_pass(img, sigma)


def show_high(high):
    '''For visualizing the high frequency image: add 128 so zero becomes grey
    (bright = positive, dark = negative), then convert to a PIL image.'''
    return Image.fromarray(to_uint8(high + 128))


def make_hybrid(img_low, img_high, sigma):
    '''Hybrid image = low frequencies of img_low + high frequencies of img_high.
    Returns the hybrid as a PIL image.'''
    low = low_pass(img_low, sigma)
    high = high_pass(img_high, sigma)   # the real high freq image, NOT the +128 one
    hybrid = low + high
    # clamp every channel to 0-255, this gets rid of most of the speckle pixels
    hybrid = np.clip(hybrid, 0, 255)
    return Image.fromarray(hybrid.astype(np.uint8))


def show_hybrid_scales(hybrid, scales=(1, 0.5, 0.25)):
    '''Shows the hybrid at a few sizes. Looking at the small ones is like
    looking at the image from further away (you mostly see the low freq image).'''
    ims = []
    for s in scales:
        w, h = int(hybrid.width * s), int(hybrid.height * s)
        ims.append(hybrid.resize((w, h), Image.BICUBIC))
    show_row(ims)

## Q1. Low frequencies (Gaussian blurred image)

I used sigma = 7 here. Smaller than that and a lot of the detail was still there, much bigger and the dog just turned into a blob.

In [ ]:
sigma_demo = 7
dog_rgb = load_rgb(pairs[0][0])

dog_low = low_pass(dog_rgb, sigma_demo)
dog_low_img = Image.fromarray(to_uint8(dog_low))
dog_low_img.save('hybrid_low.png')

print(f"left: original ({pairs[0][0]}),  right: low frequencies (sigma = {sigma_demo})")
show_row([Image.fromarray(dog_rgb.astype(np.uint8)), dog_low_img])

## Q2. High frequencies

Same sigma as above (7). The high frequency image is shown with 128 added, so grey means 0, bright is positive and dark is negative.

In [ ]:
cat_rgb = load_rgb(pairs[0][1])

cat_high = high_pass(cat_rgb, sigma_demo)
cat_high_img = show_high(cat_high)   # +128 only for showing it
cat_high_img.save('hybrid_high.png')

print(f"left: original ({pairs[0][1]}),  right: high frequencies + 128 (sigma = {sigma_demo})")
show_row([Image.fromarray(cat_rgb.astype(np.uint8)), cat_high_img])

## Q3. Hybrid images

Hybrid = low frequencies of the first image + high frequencies (without the +128) of the second one, then clamped to 0-255 per channel. Below are 3 pairs, each with 3 different sigmas. For every result I show the low frequencies, the high frequencies (+128 for visualization), and the hybrid, and after that the hybrid at a few smaller sizes (the smaller it gets, the more you see the first image instead of the second).

In [ ]:
sigmas = [4, 7, 10]

for low_file, high_file in pairs:
    img_low = load_rgb(low_file)
    img_high = load_rgb(high_file)
    # the pairs are already aligned, but they have to be the same size to add them
    assert img_low.shape == img_high.shape, "images in a pair need to be the same size"

    for s in sigmas:
        print("=" * 70)
        print(f"{low_file} (low)  +  {high_file} (high),  sigma = {s}")
        print("left to right: low frequencies, high frequencies (+128), hybrid")
        low_img = Image.fromarray(to_uint8(low_pass(img_low, s)))
        high_img = show_high(high_pass(img_high, s))
        hybrid = make_hybrid(img_low, img_high, s)
        hybrid.save(f"hybrid_{low_file.split('.')[0]}_{high_file.split('.')[0]}_sigma{s}.png")
        show_row([low_img, high_img, hybrid])

        print("hybrid at 100%, 50% and 25% size:")
        show_hybrid_scales(hybrid)

**What I noticed:** with a small sigma (4) not much gets removed from the first image, so its edges fight with the second image and up close you can see both pretty clearly. With a large sigma (10) the first image is only a blur and a lot of the second image's structure is in the high frequencies, so up close the second image dominates and you only see the first one when the image is small (or you step back). A sigma around 7 was the best balance for the pairs I tried.

# Part 4: Playing with Different Denoising Filters

## Q1. Finding good parameters

I load the images with `cv2.imread` (it gives back BGR), filter with OpenCV, and convert BGR -> RGB only for displaying with PIL. For each filter I tried a few settings first and then picked the one that looked best to me.

Parameters:
- `cv2.GaussianBlur(img, ksize, sigmaX)`: kernel size (odd) and the std of the gaussian.
- `cv2.bilateralFilter(img, d, sigmaColor, sigmaSpace)`: d = neighbourhood diameter, sigmaColor = how different an intensity can be and still get averaged in, sigmaSpace = spatial std.
- `cv2.medianBlur(img, ksize)`: kernel size (odd).

In [ ]:
img_gauss = cv2.imread('box_gauss.png')
img_speckle = cv2.imread('box_speckle.png')
# imread doesn't throw an error if the file is missing, it just returns None
assert img_gauss is not None and img_speckle is not None, "couldn't find the box images"


def cv2_to_pil(img):
    '''OpenCV images are BGR, PIL wants RGB, so swap the channels before showing.'''
    return Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))


print("left: box_gauss.png,  right: box_speckle.png")
show_row([cv2_to_pil(img_gauss), cv2_to_pil(img_speckle)])

In [ ]:
# trying out some parameters for each filter on both images.
# each row is left to right in the same order as the list printed above it

gauss_params = [((3, 3), 1), ((5, 5), 1.5), ((7, 7), 2), ((9, 9), 3)]
bilateral_params = [(5, 50, 50), (9, 75, 75), (9, 100, 75), (9, 150, 75)]
median_params = [3, 5, 7]

for name, img in [('box_gauss', img_gauss), ('box_speckle', img_speckle)]:
    print("#" * 70)
    print(name)

    print("GaussianBlur (ksize, sigmaX):", gauss_params)
    show_row([cv2_to_pil(cv2.GaussianBlur(img, ksize=k, sigmaX=s)) for k, s in gauss_params])

    print("bilateralFilter (d, sigmaColor, sigmaSpace):", bilateral_params)
    show_row([cv2_to_pil(cv2.bilateralFilter(img, d, sigmaColor=sc, sigmaSpace=ss))
              for d, sc, ss in bilateral_params])

    print("medianBlur (ksize):", median_params)
    show_row([cv2_to_pil(cv2.medianBlur(img, k)) for k in median_params])

**Best parameters I found** (picked by looking at the results above, zoomed in):

| image | GaussianBlur | bilateralFilter | medianBlur |
|---|---|---|---|
| box_gauss.png | ksize = (5, 5), sigmaX = 1.5 | d = 9, sigmaColor = 100, sigmaSpace = 75 | ksize = 5 |
| box_speckle.png | ksize = (7, 7), sigmaX = 2 | d = 9, sigmaColor = 150, sigmaSpace = 75 | ksize = 5 |

For the speckle image I needed a bit more smoothing for the Gaussian, and a bigger sigmaColor for the bilateral filter, otherwise the speckles are so different from their neighbours that the bilateral filter treats them like edges and keeps them.

In [ ]:
# the best settings from the table above
best = {
    'box_gauss':   {'gauss': ((5, 5), 1.5), 'bilateral': (9, 100, 75), 'median': 5},
    'box_speckle': {'gauss': ((7, 7), 2),   'bilateral': (9, 150, 75), 'median': 5},
}

for name, img in [('box_gauss', img_gauss), ('box_speckle', img_speckle)]:
    p = best[name]
    k, s = p['gauss']
    d, sc, ss = p['bilateral']

    out_g = cv2.GaussianBlur(img, ksize=k, sigmaX=s)
    out_b = cv2.bilateralFilter(img, d, sigmaColor=sc, sigmaSpace=ss)
    out_m = cv2.medianBlur(img, p['median'])

    # save them too so I can zoom in on them properly
    cv2.imwrite(f'{name}_best_gaussian.png', out_g)
    cv2.imwrite(f'{name}_best_bilateral.png', out_b)
    cv2.imwrite(f'{name}_best_median.png', out_m)

    print("=" * 70)
    print(f"{name}: best results")
    print(f"left to right: GaussianBlur(ksize={k}, sigmaX={s}), "
          f"bilateralFilter(d={d}, sigmaColor={sc}, sigmaSpace={ss}), medianBlur(ksize={p['median']})")
    show_row([cv2_to_pil(out_g), cv2_to_pil(out_b), cv2_to_pil(out_m)])

## Q2. The given parameters

In [ ]:
def zoom(img, factor=3):
    '''Blows up the top-left corner of an image with nearest neighbour so the artifacts
    are easier to see in the pdf (nearest so it doesn't smooth anything itself).'''
    h, w = img.shape[:2]
    crop = img[:h // 2, :w // 2]
    return cv2.resize(crop, None, fx=factor, fy=factor, interpolation=cv2.INTER_NEAREST)


for name, img in [('box_gauss', img_gauss), ('box_speckle', img_speckle)]:
    out_g = cv2.GaussianBlur(img, ksize=(7, 7), sigmaX=50)
    out_b = cv2.bilateralFilter(img, 7, sigmaColor=150, sigmaSpace=150)
    out_m = cv2.medianBlur(img, 7)

    print("=" * 70)
    print(f"{name}")
    print("left to right: original, GaussianBlur((7,7), sigmaX=50), "
          "bilateralFilter(7, 150, 150), medianBlur(7)")
    show_row([cv2_to_pil(img), cv2_to_pil(out_g), cv2_to_pil(out_b), cv2_to_pil(out_m)])

    print("same thing zoomed in (top left quarter, 3x):")
    show_row([cv2_to_pil(zoom(im)) for im in [img, out_g, out_b, out_m]])

**Pros and cons of each filter:**

**Gaussian filter** (`ksize=(7,7), sigmaX=50`)
- Pros: simple and fast, and it does reduce Gaussian noise in the flat areas since it's just a weighted average of the neighbours.
- Cons: with sigma = 50 in a 7 x 7 window the weights are basically all the same, so it acts almost like a 7 x 7 box filter. It blurs everything the same way, including the edges of the box, so they get soft and smeared. On the speckle image it doesn't really remove the speckles either, it just spreads each one out into a blurry blob, because a bright outlier pixel still has a big effect on the average.

**Bilateral filter** (`d=7, sigmaColor=150, sigmaSpace=150`)
- Pros: it only really averages pixels that have a similar intensity, so it smooths the noise inside the flat areas while keeping the strong edges of the box a lot sharper than the Gaussian does.
- Cons: it leaves quite a bit of noise behind. A very noisy pixel is very different from its neighbours, so the neighbours get a small weight and the pixel mostly keeps its own value. Zoomed in you can see that grainy leftover noise, especially right next to the edges and on the speckle image, where a lot of the speckles are still there. With sigmaColor = 150 some of the weaker details also get smoothed away, and the flat areas start to look a bit blotchy/"cartoony". It's also the slowest of the three filters.

**Median filter** (`ksize=7`)
- Pros: best one for the speckle noise. The speckles are outliers, and the median just ignores outliers, so they disappear almost completely instead of being smeared. It also keeps straight edges sharp because it doesn't create new in-between values.
- Cons: a 7 x 7 window is big, so zoomed in the corners of the box get rounded off, thin lines / small details get removed, and the image gets a blotchy, painted look. For the Gaussian noise it's not better than averaging since every pixel is a bit noisy (not just a few outliers).

Overall: Gaussian is fine if you only care about removing Gaussian noise and don't mind blur, bilateral is the best at keeping edges, and median is the best for speckle / impulse-like noise.